# DeepSceneLoc — Kaggle Training: ResNet-50 (T4 x2)

Single-model notebook — trains **ResNet-50** only (full fine-tune pipeline, same settings as the combined notebook).

Estimated: ~20-25 min/epoch, 40 epochs = ~13-16 h total -> needs 2 sessions (12 h session cap; use `--auto-resume`).

**Before running:**
1. Notebook settings → **Accelerator = GPU T4 x2**, **Internet = ON**.
2. **Add Input** → Add dataset: https://www.kaggle.com/datasets/nickj26/places2-mit-dataset
3. Make sure the `dev-krishan` branch is pushed to GitHub (this notebook clones it).


## Clone code + install deps

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/DeepSceneLoc/DeepSceneLoc-Visual-Scene-Understanding-for-Image-Based-Location-Estimation-Using-Deep-Learning.git"
BRANCH   = "dev-krishan"

import os
if not os.path.isdir("repo"):
    !git clone -b $BRANCH $REPO_URL repo
%cd repo

# torch/torchvision are preinstalled on the Kaggle GPU image; add the rest.
!pip install -q timm scikit-learn seaborn plotly google-generativeai

Cloning into 'repo'...
remote: Enumerating objects: 865, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 865 (delta 32), reused 35 (delta 28), pack-reused 810 (from 1)
Receiving objects: 100% (865/865), 18.14 MiB | 25.87 MiB/s, done.
Resolving deltas: 100% (436/436), done.
/kaggle/working/repo


## Cell 2 — Sanity check: GPUs + repo layout

In [2]:
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))

!ls scripts/training/
print("--- available Kaggle inputs ---")
!ls /kaggle/input

CUDA: True | GPUs: 2
  0 Tesla T4
  1 Tesla T4
README.md	      run_training_efficientnet_b0.py  run_training_vit_b16.py
run_ensemble_eval.py  run_training_resnet50.py
--- available Kaggle inputs ---
datasets


## Cell 3 — Prepare data

**Dataset:** Places2 MIT Dataset (https://www.kaggle.com/datasets/nickj26/places2-mit-dataset)

Set `DATASET_DIR` to the dataset path. The Places2 dataset has 365 categories, but we only need 5:
- Coastal (beach, harbor, coast, etc.)
- Forest (forest, jungle, woodland, etc.)
- Mountain (mountain, peak, valley, etc.)
- Rural (farm, countryside, village, etc.)
- Urban (city, street, building, etc.)

Two cases:
- **Already filtered to 5 categories** with split into `train/ val/ test/<Category>/` → point `DATA` directly at it.
- **Full Places2 dataset** → You'll need to filter and map categories first (use your data preparation script).

Expected final layout: `data/processed/{train,val,test}/{Coastal,Forest,Mountain,Rural,Urban}/`

In [3]:
# For Places2 MIT dataset:
DATASET_DIR = "/kaggle/input/datasets/nickj26/places2-mit-dataset"

# If you have a pre-filtered subset with only 5 categories:
# DATASET_DIR = "/kaggle/input/places2-mit-dataset/filtered_5_categories"

import os, pathlib

def _has_splits(p):
    return all(os.path.isdir(os.path.join(p, s)) for s in ("train", "val", "test"))

if _has_splits(DATASET_DIR):
    DATA = DATASET_DIR
    print("Dataset already split. Using DATA =", DATA)
else:
    DATA = "data/processed"
    print("Splitting organised folders -> data/processed ...")
    print("Reorganizing raw Places365 into super-categories...")
    !python scripts/reorganize_places365.py --data "$DATASET_DIR/train_256_places365standard/data_256" --out data/processed_raw --copy
    print("Splitting organised folders -> data/processed ...")
    !python scripts/split_dataset.py --data data/processed_raw --out data/processed --train 0.97 --val 0.02 --test 0.01 --copy

# quick counts
for split in ("train", "val", "test"):
    root = pathlib.Path(DATA) / split
    if root.exists():
        n = sum(1 for _ in root.rglob("*.jpg")) + sum(1 for _ in root.rglob("*.png"))
        print(f"  {split}: {n} images")
print("DATA =", DATA)

Splitting organised folders -> data/processed ...
Reorganizing raw Places365 into super-categories...
Scanning /kaggle/input/datasets/nickj26/places2-mit-dataset/train_256_places365standard/data_256 for Places365 categories...
Scanning categories: 100%|██████████████████████| 24/24 [00:13<00:00,  1.76it/s]
  Urban: 109330 images
  Rural: 120000 images
  Coastal: 55000 images
  Mountain: 60000 images
  Forest: 45000 images

Copying 389330 files...
  Using 32 parallel threads...
Copying files: 100%|████████████████| 389330/389330 [03:12<00:00, 2020.89file/s]
  Copied 389330 files.

Reorganization Complete!
Output directory: data/processed_raw

Images per super-category:
  Urban: 109330 images
  Rural: 120000 images
  Coastal: 55000 images
  Mountain: 60000 images
  Forest: 45000 images

Ignored (unmapped) categories: 256
Total images: 389330
Splitting organised folders -> data/processed ...
Splitting categories: 100%|███████████████████████| 5/5 [03:35<00:00, 43.18s/it]

Split statistics

## Smoke test FIRST (≈1 min)

Catches data/path/config bugs before you burn a 9-hour run. Runs 2 epochs × 5 batches.

In [4]:
!python scripts/training/run_training_resnet50.py --dry-run --batch 8 --workers 2 --data "$DATA"

  GPU 0 : Tesla T4  (14.6 GB VRAM)
  GPU 1 : Tesla T4  (14.6 GB VRAM)
  Detected 2 GPUs -- DataParallel will be used unless --no-multi-gpu is set

DeepSceneLoc - ResNet-50 Training
  Data    : data/processed
  Epochs  : 20
  Batch   : 8
  LR      : 0.001
  Workers : 2
  Patience: 8
  MinDelta: 0.001
  Device  : cuda
  Results : results/2026-07-18_11-27-44
  Dry-run : True

[Step 1/4] Validating preprocessing pipeline ...
Testing Preprocessing Pipeline

1. Testing transforms...
   Train transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=(-0.1, 0.1))
    RandomAffine(degrees=[0.0, 0.0], translate=(0.1, 0.1), scale=(0.9, 1.1))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
   Test transform: Compo

## Train ResNet-50 baseline (discriminative LR)

In [5]:
!python scripts/training/run_training_resnet50.py \
    --data "$DATA" --batch 256 --epochs 40 --workers 4 \
    --full-finetune --no-multi-gpu

  GPU 0 : Tesla T4  (14.6 GB VRAM)
  GPU 1 : Tesla T4  (14.6 GB VRAM)
  Detected 2 GPUs -- DataParallel will be used unless --no-multi-gpu is set

DeepSceneLoc - ResNet-50 Training
  Data    : data/processed
  Epochs  : 40
  Batch   : 256
  LR      : 0.001
  Workers : 4
  Patience: 8
  MinDelta: 0.001
  Device  : cuda
  Results : results/2026-07-18_11-28-27
  Dry-run : False

[Step 1/4] Validating preprocessing pipeline ...
Testing Preprocessing Pipeline

1. Testing transforms...
   Train transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=(-0.1, 0.1))
    RandomAffine(degrees=[0.0, 0.0], translate=(0.1, 0.1), scale=(0.9, 1.1))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
   Test transform: Co

## Cell 9 — Collect artifacts (→ notebook Output)

Zips checkpoints, results, and logs into `/kaggle/working/`. Download them from the **Output** tab after the run, or via `kaggle kernels output <user>/<kernel>`.

In [6]:
import shutil, os, glob

for name in ["models", "results", "logs"]:
    if os.path.isdir(name):
        out = shutil.make_archive(f"/kaggle/working/{name}", "zip", ".", name)
        print("zipped ->", out, f"({os.path.getsize(out)/1e6:.1f} MB)")
    else:
        print("skip (missing):", name)

print("\nKey files to download:")
for pat in ["models/checkpoints/**/*.pth",
            "results/**/metrics/*.json",
            "results/ensemble_*/ensemble_evaluation.json",
            "logs/**/*_history.json"]:
    for f in glob.glob(pat, recursive=True):
        print("  ", f)

zipped -> /kaggle/working/models.zip (817.4 MB)
zipped -> /kaggle/working/results.zip (5.4 MB)
zipped -> /kaggle/working/logs.zip (0.2 MB)

Key files to download:
   models/checkpoints/best_model.pth
   models/checkpoints/checkpoint_epoch_10.pth
   models/checkpoints/checkpoint_epoch_5.pth
   results/metrics/resnet50_evaluation.json
   results/2026-03-13_15-58-44/metrics/resnet50_evaluation.json
   results/EfficientNet-B0_20260428_134645/metrics/DeepSceneLocEfficientNetAdvanced_evaluation.json
   results/EfficientNet-B0_20260428_005152/metrics/DeepSceneLocEfficientNetAdvanced_evaluation.json
   results/2026-03-15_13-48-03/metrics/resnet50_evaluation.json
   results/2026-07-18_11-28-27/metrics/resnet50_evaluation.json
   results/2026-03-13_10-49-17/metrics/resnet50_evaluation.json
   results/ViT-B_16_20260505_155758/metrics/DeepSceneLocViTAdvanced_evaluation.json
   logs/training_history.json


## Resume across sessions (12 h limit)

If the run gets cut off, start a new session, re-run the setup cells, then re-run training **with** `--auto-resume` — it finds the latest checkpoint and continues:

```bash
!python scripts/training/run_training_resnet50.py --data "$DATA" --batch 256 --epochs 40 --workers 4 --full-finetune --auto-resume
```

Checkpoints must survive between sessions: either **commit** the notebook (saves `/kaggle/working`) or persist `models/` to a Kaggle Dataset you re-attach next session.
